# Ion template: 离子状态、Nernst 方程和 ODE

这个 notebook 解释当前 `braincell` 中 ion template 如何把离子浓度和反转电位组织起来，并说明它们如何对应数学公式。

当前共享模板在 `braincell.ion._base` 中：

- `FixedIon`：固定 `Ci/Co/E/valence`，没有浓度 ODE。
- `InitNernstIon`：固定 `Ci/Co`，在初始化或 reset 时根据 Nernst 方程更新 `E`。
- `DynamicNernstIon`：`Ci` 是 `DiffEqState`，`E` 是当前 `Ci` 的 Nernst 派生属性，`derivative(...)` 定义 `dCi/dt`。

这里以仓库里的 `CalciumDetailed` 为主要例子，因为它同时包含动态浓度、总钙电流和 Nernst 反转电位。

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import brainunit as u
import jax.numpy as jnp

import braincell
from braincell import Branch, Cell, Morphology
from braincell.filter import BranchSlice
from braincell.mech import Channel, Ion
from braincell.ion import CalciumDetailed, CalciumFixed, CalciumInitNernst, PotassiumFixed

print("braincell version:", braincell.__version__)

## 1. Ion 在通道和细胞之间传什么？

`Ion.pack_info()` 会把离子状态打包成 `IonInfo`：

```python
IonInfo(Ci=..., Co=..., E=..., valence=...)
```

channel 的 `current(V, ion_info)` 不需要直接知道 ion 对象本身，只需要这些物理量。典型电流写法是：

$$
I_x = g_x(V, \mathrm{state})\,(E_x - V)
$$

其中 `E_x`、`Ci`、`Co` 来自 ion。

In [ ]:
k = PotassiumFixed(size=1, E=-90.0 * u.mV)
info = k.pack_info()

print("Ci:", info.Ci)
print("Co:", info.Co)
print("E:", info.E)
print("valence:", info.valence)

## 2. `FixedIon` 和 `InitNernstIon`

`FixedIon` 的数学含义最简单：

$$
\frac{dC_i}{dt}=0,\qquad E=E_{fixed}
$$

`InitNernstIon` 不让 `Ci` 动态变化，但 `init_state()` 和 `reset_state()` 会用 Nernst 方程计算并保存 `E`：

$$
E = \frac{RT}{zF}\log\frac{C_o}{C_i}
$$

这适合“浓度固定，但反转电位想由浓度和温度推导”的场景。

In [ ]:
ca_fixed = CalciumFixed(size=1, E=120.0 * u.mV)
ca_nernst = CalciumInitNernst(size=1, temp=u.celsius2kelvin(36.0))

V = jnp.array([-65.0]) * u.mV
ca_nernst.init_state(V)

expected_E = (
    u.gas_constant * ca_nernst.temp / (ca_nernst.valence * u.faraday_constant)
    * u.math.log(ca_nernst.Co / ca_nernst.Ci)
)

print("CalciumFixed E:", ca_fixed.E)
print("CalciumInitNernst E:", ca_nernst.E.to_decimal(u.mV), "mV")
print("Nernst formula E:", expected_E.to_decimal(u.mV), "mV")

## 3. `DynamicNernstIon`: `Ci` 是 ODE 状态

`DynamicNernstIon` 把 `Ci` 创建为 `DiffEqState`，所以它有：

- `Ci.value`：当前胞内浓度。
- `Ci.derivative`：积分器要用的 `dCi/dt`。
- `E`：由当前 `Ci.value` 实时计算出来的 Nernst 反转电位。

template 本身不规定具体 `dCi/dt`，而是调用具体 ion 的 `derivative(Ci, V, total_current=...)`。如果 `uses_total_current=True`，template 会先把该 ion 下所有 channel 的电流求和，再传给 `derivative(...)`。

## 4. 仓库例子：`CalciumDetailed`

`CalciumDetailed` 在当前源码中继承 `Calcium` 和 `DynamicNernstIon`。它的浓度 ODE 是：

$$
\frac{d[Ca]_i}{dt}
= \max\left(\frac{I_{Ca}}{2Fd}, 0\right)
+ \frac{[Ca]_{rest} - [Ca]_i}{\tau}
$$

同时反转电位实时由 Nernst 方程给出：

$$
E_{Ca} = \frac{RT}{zF}\log\frac{[Ca]_o}{[Ca]_i}
$$

这里 `tau` 是钙回到 resting concentration 的时间常数；`d` 是膜下薄壳厚度；`I_Ca` 是该 calcium ion 容器里所有 calcium channel 的总电流。

In [ ]:
ca = CalciumDetailed(
    size=1,
    tau=5.0 * u.ms,
    C_rest=1.0e-4 * u.mM,
    Ci_initializer=6.0e-4 * u.mM,
)
ca.init_state(V)

# 这个 cell 不挂载 calcium channel，所以把 total_current 显式设为 0，聚焦 tau 项。
zero_current = jnp.zeros(ca.varshape) * (u.uA / u.cm**2)
dci = ca.derivative(ca.Ci.value, V, total_current=zero_current)
expected_dci = (ca.C_rest - ca.Ci.value) / ca.tau

print("Ci:", ca.Ci.value.to_decimal(u.mM), "mM")
print("E(Ci):", ca.E.to_decimal(u.mV), "mV")
print("dCi/dt:", dci.to_decimal(u.mM / u.ms), "mM/ms")
print("tau term:", expected_dci.to_decimal(u.mM / u.ms), "mM/ms")

`DynamicNernstIon.E` 是属性，不是固定缓存。改变 `Ci.value` 后，`E` 会按 Nernst 方程立即变化。

In [ ]:
e_low_ci = ca.E
ca.Ci.value = jnp.array([1.0e-2]) * u.mM
e_high_ci = ca.E

print("E at low Ci:", e_low_ci.to_decimal(u.mV), "mV")
print("E after raising Ci:", e_high_ci.to_decimal(u.mV), "mV")

## 5. 在 multi-compartment `Cell` 里使用 ion template

在 multi-compartment 层，`mech.Ion(...)` 是声明，`Cell.init_state()` 会把它降低为 runtime ion 对象。下面例子把 `CalciumDetailed` paint 到一个简单 morphology 上，再给它绑定一个 `CaT_HM1992` channel。

这展示了两个关系：

- `Ion("CalciumDetailed", name="ca_dyn", ...)` 创建动态 calcium ion 容器。
- `Channel("CaT_HM1992", ion_name="ca_dyn", ...)` 绑定到这个 ion，并贡献 `I_Ca`。

In [ ]:
def build_demo_morphology():
    soma = Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[8.0, 8.0] * u.um,
        type="soma",
    )
    dend = Branch.from_lengths(
        lengths=[60.0] * u.um,
        radii=[2.0, 1.2] * u.um,
        type="basal_dendrite",
    )
    morpho = Morphology.from_root(soma, name="soma")
    morpho.attach(parent="soma", child_branch=dend, child_name="dend", parent_x=1.0)
    return morpho


cell = Cell(build_demo_morphology())
region = BranchSlice(branch_index=[0, 1], prox=0.0, dist=1.0)

cell.paint(
    region,
    Ion(
        "CalciumDetailed",
        name="ca_dyn",
        d=0.5 * u.um,
        tau=10.0 * u.ms,
        C_rest=5.0e-5 * u.mM,
        Ci_initializer=2.4e-4 * u.mM,
    ),
)
cell.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Channel("CaT_HM1992", ion_name="ca_dyn", g_max=2.0 * (u.mS / u.cm**2)),
)

cell.init_state()
ion = cell.get_ion("ca_dyn")
layout = next(layout for layout in cell.layouts if layout.kind == "ion:CalciumDetailed")

print("layout id:", layout.id)
print("runtime ion type:", type(ion).__name__)
print("Ci is DiffEqState:", type(ion.Ci).__name__)
print("initial Ci shape:", ion.Ci.value.shape)
print("initial E at first CV:", float(ion.E[0].to_decimal(u.mV)), "mV")

In [ ]:
cell.compute_derivative()

ca_channel = ion.channels["CaT_HM1992"]
print("CaT p derivative shape:", ca_channel.p.derivative.shape)
print("CaT q derivative shape:", ca_channel.q.derivative.shape)
print("Ca Ci derivative shape:", ion.Ci.derivative.shape)
print("Ca Ci derivative first CV:", ion.Ci.derivative[0].to_decimal(u.mM / u.ms), "mM/ms")

## Summary

- `Ion` 负责管理一类 ion 的 `Ci/Co/E/valence`，并把这些值以 `IonInfo` 传给 channel。
- `FixedIon` 没有 ODE，适合固定反转电位或固定浓度场景。
- `InitNernstIon` 用固定浓度初始化 Nernst 反转电位。
- `DynamicNernstIon` 把 `Ci` 变成可积分状态，并把具体 `dCi/dt` 留给 concrete ion。
- `CalciumDetailed` 是当前仓库中动态 ion template 的直接例子：`Ci` 是 ODE 状态，`E` 由 Nernst 方程实时计算，`tau` 控制浓度回到 resting value 的速度。